# Task 2 — fixed probability ensemble check (v9)

Test the instruction slide's stacking/ensemble hint on the same five development folds used in earlier rounds. This is a **prespecified soft ensemble**, not a learned stack: 50% CatBoost + 50% LightGBM, and equal thirds CatBoost/LightGBM/XGBoost. Reuse saved CatBoost OOF probabilities; refit only the two missing model families in each fold. The repeatedly used holdout and Test are not consulted. An ensemble becomes a submission candidate only if it beats the tuned weighted CatBoost CV reference by a practical margin on paired folds. Saved base-model fits already exist from earlier experiments; this notebook creates no new ensemble artifact when the fixed combinations lose.

In [1]:
from pathlib import Path
import sys, json, hashlib, joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_model.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.pipeline.random_seed import set_seed
from src.task2.models.decision_rules import restore_categorical_nulls
SEED=set_seed()
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
DATA='eda_original_v2'; SRC=ROOT/'src'; TASK=SRC/'task2'; RESULTS=TASK/'models'/'results'
card=json.loads((TASK/'data'/'eda'/f'{DATA}.json').read_text(encoding='utf-8'))
train=pd.read_parquet(SRC/card['splits']['train']['path'])
y=train.segment.astype(int)
raw=train.drop(columns=['player_id','segment'])
X=restore_categorical_nulls(raw,raw.select_dtypes(include=['object']).columns.tolist())
dev_pos,_=train_test_split(np.arange(len(y)),test_size=.2,stratify=y,random_state=SEED)
X_dev=X.iloc[dev_pos].reset_index(drop=True); y_dev=y.iloc[dev_pos].reset_index(drop=True)
folds=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(X_dev,y_dev))
assert len(X_dev)==83200 and len(folds)==5
fold_number=np.empty(len(y_dev),dtype=np.int8)
for i,(_,check) in enumerate(folds,1):fold_number[check]=i
print('Development rows:',len(y_dev))

Development rows: 83200


## 1. Reconstruct saved CatBoost probabilities

The earlier baseline and weighted CatBoost OOF predictions have exactly the same row positions and folds. Validate that alignment before blending.

In [2]:
def saved_oof(stem,version):
    parts=[pd.read_parquet(RESULTS/f'{stem}_fold{i}_{version}.parquet') for i in range(1,6)]
    frame=pd.concat(parts,ignore_index=True).sort_values('row_pos').reset_index(drop=True)
    assert np.array_equal(frame.row_pos,np.arange(len(y_dev)))
    assert np.array_equal(frame.fold,fold_number)
    assert np.array_equal(frame.true,y_dev)
    return frame[[f'p_{j}' for j in range(4)]].to_numpy()
cat_oof=saved_oof('class_bias','v1')
weighted_cat_oof=saved_oof('weighted_23','v4')
def scores(prob):
    pred=prob.argmax(axis=1)
    return np.array([f1_score(y_dev.iloc[check],pred[check],average='macro') for _,check in folds])
print('CatBoost baseline:',scores(cat_oof).round(6),'mean',scores(cat_oof).mean())
print('CatBoost weighted:',scores(weighted_cat_oof).round(6),'mean',scores(weighted_cat_oof).mean())

CatBoost baseline: [0.756123 0.746062 0.750865 0.758046 0.751726] mean 0.7525642419662718
CatBoost weighted: [0.755899 0.747641 0.753011 0.756484 0.752465] mean 0.7531000717248977


## 2. LightGBM OOF model

Clone the previously saved LightGBM pipeline, including its preprocessing. Each fold fits preprocessing only on its training rows. Fold files are resumable and checked against the dataset and model cards.

In [3]:
def fit_oof(model_name, cache_name):
    model_card=json.loads((TASK/'save'/f'{model_name}.json').read_text(encoding='utf-8'))
    assert model_card['data_name']==DATA
    source=joblib.load(SRC/model_card['model_path'])
    key={'dataset':DATA,'dataset_created_at':card['created_at'],'model':model_name,
         'model_created_at':model_card['created_at'],'seed':SEED,
         'dev_hash':hashlib.sha256(np.asarray(dev_pos,dtype=np.int64).tobytes()).hexdigest()}
    parts=[]
    for i,(fit,check) in enumerate(folds,1):
        path=RESULTS/f'{cache_name}_fold{i}.parquet'; meta=path.with_suffix('.json')
        if path.exists()!=meta.exists():raise RuntimeError(f'Incomplete fold: {path}')
        if path.exists():
            assert json.loads(meta.read_text(encoding='utf-8'))=={'key':key,'fold':i}
            part=pd.read_parquet(path)
        else:
            model=clone(source);model.fit(X_dev.iloc[fit],y_dev.iloc[fit])
            prob=np.asarray(model.predict_proba(X_dev.iloc[check]))
            part=pd.DataFrame(prob,columns=[f'p_{j}' for j in range(4)])
            part.insert(0,'row_pos',check);part.insert(0,'fold',i)
            part.to_parquet(path,index=False)
            meta.write_text(json.dumps({'key':key,'fold':i},indent=2),encoding='utf-8')
            print(model_name,'fold',i,'fitted',flush=True)
        assert np.array_equal(part.row_pos,check)
        assert np.allclose(part[[f'p_{j}' for j in range(4)]].sum(axis=1),1)
        parts.append(part)
    frame=pd.concat(parts,ignore_index=True).sort_values('row_pos')
    assert np.array_equal(frame.row_pos,np.arange(len(y_dev)))
    return frame[[f'p_{j}' for j in range(4)]].to_numpy()
lgb_oof=fit_oof('t2_lgb_original_no_noise_v2','t2_lgb_ensemble_cv5_v9')
print('LightGBM:',scores(lgb_oof).round(6),'mean',scores(lgb_oof).mean())

t2_lgb_original_no_noise_v2 fold 1 fitted


t2_lgb_original_no_noise_v2 fold 2 fitted


t2_lgb_original_no_noise_v2 fold 3 fitted


t2_lgb_original_no_noise_v2 fold 4 fitted


t2_lgb_original_no_noise_v2 fold 5 fitted


LightGBM: [0.750865 0.743219 0.746601 0.749848 0.74522 ] mean 0.7471506580620225


## 3. XGBoost OOF model

Use the same paired folds, source pipeline, and cache checks.

In [4]:
xgb_oof=fit_oof('t2_xgb_original_no_noise_v2','t2_xgb_ensemble_cv5_v9')
print('XGBoost:',scores(xgb_oof).round(6),'mean',scores(xgb_oof).mean())

t2_xgb_original_no_noise_v2 fold 1 fitted


t2_xgb_original_no_noise_v2 fold 2 fitted


t2_xgb_original_no_noise_v2 fold 3 fitted


t2_xgb_original_no_noise_v2 fold 4 fitted


t2_xgb_original_no_noise_v2 fold 5 fitted


XGBoost: [0.749054 0.743026 0.744878 0.751354 0.749204] mean 0.7475031794025639


## 4. Prespecified ensemble comparison

Use fixed weights. No grid or learned meta-model is fitted on these OOF labels. The tuned weighted CatBoost v7 mean is a same-fold reference but its fold probabilities were not saved, so compare its recorded fold scores rather than claiming per-row paired error changes.

In [5]:
v7=json.loads((RESULTS/'t2_cat_tuned_weighted_cv5_v7.json').read_text(encoding='utf-8'))
v7_scores=np.array([r['macro_F1'] for r in sorted(v7['rows'],key=lambda r:r['fold'])])
variants={'original_cat':cat_oof,'weighted_cat':weighted_cat_oof,
          'lightgbm':lgb_oof,'xgboost':xgb_oof,
          'cat_lgb_half':.5*cat_oof+.5*lgb_oof,
          'weighted_cat_lgb_half':.5*weighted_cat_oof+.5*lgb_oof,
          'cat_lgb_xgb_thirds':(cat_oof+lgb_oof+xgb_oof)/3,
          'weighted_cat_lgb_xgb_thirds':(weighted_cat_oof+lgb_oof+xgb_oof)/3}
table=pd.DataFrame({name:scores(prob) for name,prob in variants.items()},index=np.arange(1,6))
table.insert(0,'tuned_weighted_cat_v7',v7_scores)
summary=pd.DataFrame({'mean_macro_F1':table.mean(),'delta_vs_v7':table.mean()-v7_scores.mean(),
                      'fold_wins_vs_v7':table.gt(v7_scores,axis=0).sum()}).sort_values('mean_macro_F1',ascending=False)
display(table.round(6));display(summary.round(6))
RESULTS.joinpath('t2_ensemble_fixed_cv5_v9.json').write_text(json.dumps({
    'dataset':DATA,'seed':SEED,'fold_scores':{c:table[c].tolist() for c in table},
    'summary':summary.reset_index(names='model').to_dict('records'),
    'note':'Development folds only; fixed weights; v7 probabilities unavailable; no Test use'},
    indent=2)+'\n',encoding='utf-8')
print('Best fixed ensemble:',summary.loc[[n for n in summary.index if n not in ['tuned_weighted_cat_v7','original_cat','weighted_cat','lightgbm','xgboost']][0]].to_dict())

,tuned_weighted_cat_v7,original_cat,weighted_cat,lightgbm,xgboost,cat_lgb_half,weighted_cat_lgb_half,cat_lgb_xgb_thirds,weighted_cat_lgb_xgb_thirds
1,0.756632,0.756123,0.755899,0.750865,0.749054,0.754163,0.755236,0.753941,0.755907
2,0.750171,0.746062,0.747641,0.743219,0.743026,0.745744,0.746744,0.744961,0.745715
3,0.752882,0.750865,0.753011,0.746601,0.744878,0.750628,0.751588,0.749635,0.750040
4,0.757868,0.758046,0.756484,0.749848,0.751354,0.755644,0.754957,0.756480,0.755261
5,0.753229,0.751726,0.752465,0.745220,0.749204,0.751087,0.752214,0.751265,0.751350


,mean_macro_F1,delta_vs_v7,fold_wins_vs_v7
tuned_weighted_cat_v7,0.754156,0.000000,0
weighted_cat,0.753100,-0.001056,1
original_cat,0.752564,-0.001592,1
weighted_cat_lgb_half,0.752148,-0.002008,0
weighted_cat_lgb_xgb_thirds,0.751655,-0.002502,0
cat_lgb_half,0.751453,-0.002703,0
cat_lgb_xgb_thirds,0.751256,-0.002900,0
xgboost,0.747503,-0.006653,0
lightgbm,0.747151,-0.007006,0


Best fixed ensemble: {'mean_macro_F1': 0.7521479113678435, 'delta_vs_v7': -0.0020083910964767693, 'fold_wins_vs_v7': 0.0}


## Decision

Read the executed comparison above. Do not replace v8 or create a Kaggle submission for a tiny gain on already reused folds. The task remains frozen unless this experiment produces a clear, consistent improvement.